# SafeLens Phase 4 -- DeBERTa training on Colab GPU

Local M2 (MPS) training was measured at ~370-524 seconds/step (projected 53-76 hours for 525 steps) -- infeasible. This notebook runs the **exact same, unmodified** `scripts/train_deberta.py` used locally, on a Colab GPU. Device selection is automatic (`safelens.utils.device.detect_device` picks CUDA when available), so no code changes are needed for a fair, reproducible comparison against the frozen Phase 3 baseline.

**Before running:** Runtime -> Change runtime type -> GPU (T4 or better).

**You need one thing this notebook cannot fetch on its own:** a GitHub token with read access to the private `SafeLens` repo, stored as a Colab secret named `GITHUB_TOKEN` (key icon in the left sidebar). Everything else -- including the frozen Phase 2 processed split, committed as a small zip specifically so this notebook doesn't need a manual upload step -- comes from the git clone below.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!git clone https://{token}@github.com/saitejasrivilli/SafeLens.git
%cd SafeLens

## Unzip the frozen Phase 2 processed split

`data/processed/civil_comments/civil_comments_processed.zip` is committed to the repo (an explicit `.gitignore` exception) precisely so this training run uses the *exact* same train/validation/test split Phase 3 was evaluated against -- not a regenerated one.

In [ ]:
!unzip -o data/processed/civil_comments/civil_comments_processed.zip -d data/processed/civil_comments
!ls -la data/processed/civil_comments

In [ ]:
!pip install -q -e ".[dev]"

In [ ]:
import torch
print('cuda available:', torch.cuda.is_available())
print('device name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## Run the exact same training script used locally

No code changes. `detect_device()` will select CUDA automatically.

In [ ]:
!python scripts/train_deberta.py

## Package results for download

Two archives: a small one (experiment.json, plots, config/metadata -- safe to commit) and a large one (trained model weights -- for local use only, still not committed to Git per Phase 4 instructions).

In [ ]:
!cd benchmarks/results/deberta && zip -r /content/deberta_benchmark_results.zip .
!cd models/text/deberta && zip -r /content/deberta_model_artifacts.zip v1
from google.colab import files
files.download('/content/deberta_benchmark_results.zip')
files.download('/content/deberta_model_artifacts.zip')

## After downloading

Locally: unzip `deberta_benchmark_results.zip` into `benchmarks/results/deberta/` and `deberta_model_artifacts.zip` into `models/text/deberta/` (the latter stays gitignored). Then continue the Phase 4 write-up (docs, comparison table, error analysis) from the real `experiment.json` produced here -- do not fabricate numbers if this run's results differ from any earlier expectation.